### Import Dependencies

In [87]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm
import mlflow
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    r2_score, 
    mean_absolute_error, 
    root_mean_squared_error
)

pd.set_option('display.max_columns', None)

### Data Preparation

In [88]:
# NASA S-metric: Measures the practical cost of prediction errors
# with a higher penalty for dangerous overestimation of remaining life.
def s_score(
    rul_true: float,
    rul_pred: float,
) -> float:
    """
    Compute the NASA C-MAPSS S-score.

    Parameters
    ----------
    rul_true : np.ndarray
        Ground truth Remaining Useful Life (RUL).
    rul_pred : np.ndarray
        Predicted Remaining Useful Life (RUL).

    Returns
    -------
    float
        Total NASA S-score. Lower is better.
    """

    diff = rul_pred - rul_true

    score = np.where(
        diff < 0,
        np.exp(-diff / 13.0) - 1.0,
        np.exp(diff / 10.0) - 1.0,
    )

    return float(np.sum(score))

In [89]:
# reading data from source
train = pd.read_csv(r'..\data\train_FD001.txt', sep=r'\s+', header=None)
test = pd.read_csv(r'..\data\test_FD001.txt', sep=r'\s+', header=None)
test_labels = pd.read_csv(r'..\data\RUL_FD001.txt', header=None)

# feature names as written in readme file
columns = (
    ["unit_number", "time_cycles"]
    + [f"operational_setting_{i}" for i in range(1, 4)]
    + [f"sensor_measurement_{i}" for i in range(1, 22)]
)

# renaming features
train.columns = columns
test.columns = columns
test_labels.columns = ['rul']

In [90]:
# drop null features (complete case analysis)
train = train.dropna(axis=1)
test = test.dropna(axis=1)

# dropping constant features
constant_features = [
    'operational_setting_3', 
    'sensor_measurement_1', 
    'sensor_measurement_5',
    'sensor_measurement_6',
    'sensor_measurement_10', 
    'sensor_measurement_16',
    'sensor_measurement_18',
    'sensor_measurement_19'
]

train = train.drop(constant_features, axis=1)
test = test.drop(constant_features, axis=1)

In [91]:
# maximum cycle for each engine unit
max_cycles = train.groupby('unit_number')['time_cycles'].max().reset_index()
max_cycles.columns = ['unit_number', 'max_cycle']

# merge this maximum back into the main dataframe
train = train.merge(max_cycles, on='unit_number', how='left')

# calculate the standard linear RUL
train['linear_rul'] = train['max_cycle'] - train['time_cycles']

# drop the helper column
train.drop(columns=['max_cycle'], inplace=True)

# industry standard cap for FD001
R_max = 120

# apply the ceiling cap: if linear_rul > R_max, force it to be R_max
train['piecewise_rul'] = train['linear_rul'].clip(upper=R_max)

In [92]:
# train-test split
X = train.drop(['unit_number', 'time_cycles', 'linear_rul', 'piecewise_rul', 'operational_setting_1', 'operational_setting_2'], axis=1)
y_linear = train['linear_rul']
y_piecewise = train['piecewise_rul']

X_test = (test.groupby("unit_number").tail(1))
X_test = X_test.drop(['unit_number', 'time_cycles', 'operational_setting_1', 'operational_setting_2'], axis=1)
y_test = test_labels.copy(deep=True)

print(f'Shape of X: {X.shape}')
print(f'Shape of y_linear: {y_linear.shape}')
print(f'Shape of y_piecewise: {y_piecewise.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

Shape of X: (20631, 14)
Shape of y_linear: (20631,)
Shape of y_piecewise: (20631,)
Shape of X_test: (100, 14)
Shape of y_test: (100, 1)


### Configuration

In [81]:
class Config:

    SEED = 42
    N_SPLITS = 5

    TARGET_TYPE = 'piecewise_rul'

    HISTGBM_PARAMS = {
        'categorical_features': 'from_dtype',
        'early_stopping': 'auto',
        'interaction_cst': None,
        'l2_regularization': 0.0,
        'learning_rate': 0.1,
        'loss': 'squared_error',
        'max_bins': 255,
        'max_depth': 5,
        'max_features': 0.8,
        'max_iter': 400,
        'max_leaf_nodes': 31,
        'min_samples_leaf': 20,
        'monotonic_cst': None,
        'n_iter_no_change': 10,
        'quantile': None,
        'random_state': SEED,
        'scoring': 'loss',
        'tol': 1e-07,
        'validation_fraction': None,
        'verbose': 0,
        'warm_start': False
    }

    ETREES_PARAMS = {
        'bootstrap': True,
        'ccp_alpha': 0.0,
        'criterion': 'squared_error',
        'max_depth': None,
        'max_features': 1.0,
        'max_leaf_nodes': None,
        'max_samples': None,
        'min_impurity_decrease': 0.0,
        'min_samples_leaf': 1,
        'min_samples_split': 2,
        'min_weight_fraction_leaf': 0.0,
        'monotonic_cst': None,
        'n_estimators': 400,
        'n_jobs': -1,
        'oob_score': False,
        'random_state': SEED,
        'verbose': 0,
        'warm_start': False
    }

    XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.05,
        'n_estimators': 1200,
        'max_depth': 5,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
        'early_stopping_rounds': 100,
        'min_child_weight': 3, 
        'reg_alpha': 0.1, 
        'reg_lambda': 1.0, 
        'gamma': 0.1
    }

    LGBM_PARAMS = {
        "boosting_type": "gbdt",
        "data_sample_strategy": "goss",
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.03,
        "n_estimators": 500,
        "num_leaves": 31,
        "max_depth": -1,
        "colsample_bytree": 1.0,
        "reg_alpha": 1.0,
        "reg_lambda": 0.5,
        "random_state": SEED,
        "importance_type": "gain",
        "n_jobs": -1,
    }

TARGETS = {
    "linear_rul": y_linear,
    "piecewise_rul": y_piecewise,
}

y = TARGETS[Config.TARGET_TYPE]

VERSION = "V6"

### Common utils

In [83]:
def plot_permutation_importance(
    model,
    X: pd.DataFrame,
    y,
    scoring: str = "neg_mean_squared_error",
    n_repeats: int = 10,
    top_k: int | None = None,
    random_state: int = 42,
    figsize: tuple = (10, 8),
) -> None:
    """
    Plot permutation feature importance.

    Parameters
    ----------
    model : estimator
        Trained scikit-learn model.

    X : pd.DataFrame
        Feature matrix.

    y : array-like
        Target values.

    scoring : str, default="neg_mean_squared_error"
        Scoring metric used during permutation importance.

    n_repeats : int, default=20
        Number of feature shuffles.

    top_k : int or None, default=None
        Number of top features to display.
        If None, all features are plotted.

    random_state : int, default=42
        Random seed.

    figsize : tuple, default=(10, 8)
        Figure size.
    """

    perm = permutation_importance(
        estimator=model,
        X=X,
        y=y,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1,
    )

    importance_df = (
        pd.DataFrame(
            {
                "Feature": X.columns,
                "Importance": perm.importances_mean,
                "Std": perm.importances_std,
            }
        )
        .sort_values("Importance", ascending=False)
    )

    if top_k is not None:
        importance_df = importance_df.head(top_k)

    fig, ax = plt.subplots(figsize=figsize)

    ax.barh(
        importance_df["Feature"],
        importance_df["Importance"],
        xerr=importance_df["Std"],
        color="#5B8FF9",
        edgecolor="black",
    )

    ax.invert_yaxis()

    ax.set_title(
        "Permutation Feature Importance",
        fontsize=18,
        fontweight="bold",
        pad=15,
    )

    ax.set_xlabel(
        "Mean Importance",
        fontsize=12,
    )

    ax.set_ylabel("")

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.3,
    )

    sns.despine()

    plt.tight_layout()

    return fig

In [84]:
def plot_actual_vs_predicted_scatter(
    y_true,
    y_pred,
    title: str = "Actual vs Predicted",
    figsize: tuple = (7, 7),
):
    """
    Plot Actual vs Predicted scatter plot.

    Parameters
    ----------
    y_true : array-like
        Ground truth values.

    y_pred : array-like
        Predicted values.

    title : str, default="Actual vs Predicted"
        Plot title.

    figsize : tuple, default=(7, 7)
        Figure size.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Matplotlib figure.
    """

    fig, ax = plt.subplots(figsize=figsize)

    sns.scatterplot(
        x=y_true,
        y=y_pred,
        s=30,
        alpha=0.6,
        edgecolor=None,
        ax=ax,
    )

    # Perfect prediction line
    lims = [
        min(np.min(y_true), np.min(y_pred)),
        max(np.max(y_true), np.max(y_pred)),
    ]

    ax.plot(
        lims,
        lims,
        linestyle="--",
        color="red",
        linewidth=2,
        label="Perfect Prediction",
    )

    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=15,
    )

    ax.set_xlabel(
        "Actual Remaining Useful Life",
        fontsize=12,
    )

    ax.set_ylabel(
        "Predicted Remaining Useful Life",
        fontsize=12,
    )

    ax.grid(
        alpha=0.3,
        linestyle="--",
    )

    ax.legend()

    sns.despine()

    plt.tight_layout()

    return fig

## Training and Experiment Tracking

### 1. HistGradientBoostingRegressor

In [ ]:
# MLflow
EXPERIMENT_NAME = "histgbm"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.HISTGBM_PARAMS)

    mlflow.log_params(
        {
            "model": "HistGradientBoostingRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = HistGradientBoostingRegressor(
            **Config.HISTGBM_PARAMS
        )

        # Train
        model.fit(X_train, y_train)

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 17.8454 | MAE: 12.4846 | R2 score: 0.7947 | S-score: 55913.3942
Fold 02 | RMSE: 17.3302 | MAE: 12.3201 | R2 score: 0.8114 | S-score: 44752.1635
Fold 03 | RMSE: 17.2787 | MAE: 12.2490 | R2 score: 0.8097 | S-score: 46864.7341
Fold 04 | RMSE: 16.8106 | MAE: 11.9524 | R2 score: 0.8235 | S-score: 38965.5620
Fold 05 | RMSE: 17.3336 | MAE: 12.2418 | R2 score: 0.8153 | S-score: 48194.0878

OOF RESULTS
RMSE: 17.3228
MAE: 12.2496
R2 score: 0.8111
S-score: 234689.94

TEST RESULTS
RMSE: 17.9529
MAE: 13.0286
R2 score: 0.8134
S-score: 22645615.68
Logging Plots
Logging Done


### 2. ExtraTreesRegressor

In [ ]:
# MLflow
EXPERIMENT_NAME = "etrees"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.ETREES_PARAMS)

    mlflow.log_params(
        {
            "model": "ExtraTreesRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = ExtraTreesRegressor(
            **Config.ETREES_PARAMS
        )

        # Train
        model.fit(X_train, y_train)

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 17.3623 | MAE: 12.5692 | R2 score: 0.8056 | S-score: 44845.3707
Fold 02 | RMSE: 16.9222 | MAE: 12.4190 | R2 score: 0.8202 | S-score: 36222.4944
Fold 03 | RMSE: 16.8852 | MAE: 12.3605 | R2 score: 0.8183 | S-score: 36847.2545
Fold 04 | RMSE: 16.3466 | MAE: 12.0074 | R2 score: 0.8331 | S-score: 30808.3323
Fold 05 | RMSE: 16.9726 | MAE: 12.4048 | R2 score: 0.8230 | S-score: 38952.5575

OOF RESULTS
RMSE: 16.9009
MAE: 12.3522
R2 score: 0.8202
S-score: 187676.01

TEST RESULTS
RMSE: 17.6376
MAE: 12.9668
R2 score: 0.8199
S-score: 18945147.10


### 3. XGBRegressor

In [ ]:
# MLflow
EXPERIMENT_NAME = "xgboost"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.XGB_PARAMS)

    mlflow.log_params(
        {
            "model": "XGBRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = XGBRegressor(
            **Config.XGB_PARAMS
        )

        # Train
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=0
        )

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 17.3722 | MAE: 12.2673 | R2 score: 0.8054 | S-score: 49723.4816
Fold 02 | RMSE: 16.9896 | MAE: 12.1385 | R2 score: 0.8188 | S-score: 39995.6151
Fold 03 | RMSE: 16.8911 | MAE: 12.0624 | R2 score: 0.8181 | S-score: 40574.1543
Fold 04 | RMSE: 16.3603 | MAE: 11.7744 | R2 score: 0.8329 | S-score: 33694.5394
Fold 05 | RMSE: 16.9771 | MAE: 12.0591 | R2 score: 0.8229 | S-score: 42634.8272

OOF RESULTS
RMSE: 16.9212
MAE: 12.0604
R2 score: 0.8198
S-score: 206622.62

TEST RESULTS
RMSE: 17.7134
MAE: 12.7762
R2 score: 0.8183
S-score: 21513951.10


### 4. LGBMRegressor

In [ ]:
# MLflow
EXPERIMENT_NAME = "lgbm"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.LGBM_PARAMS)

    mlflow.log_params(
        {
            "model": "LGBMRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = LGBMRegressor(
            **Config.LGBM_PARAMS
        )

        # Train
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[early_stopping(100, verbose=False)]
        )

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 17.4595 | MAE: 12.4152 | R2 score: 0.8034 | S-score: 48004.6755
Fold 02 | RMSE: 16.9651 | MAE: 12.1789 | R2 score: 0.8193 | S-score: 39087.1564
Fold 03 | RMSE: 16.8935 | MAE: 12.0803 | R2 score: 0.8181 | S-score: 39740.7800
Fold 04 | RMSE: 16.3507 | MAE: 11.7221 | R2 score: 0.8331 | S-score: 34057.9891
Fold 05 | RMSE: 17.0409 | MAE: 12.1734 | R2 score: 0.8215 | S-score: 43026.4958

OOF RESULTS
RMSE: 16.9457
MAE: 12.1140
R2 score: 0.8192
S-score: 203917.10

TEST RESULTS
RMSE: 17.8489
MAE: 12.9300
R2 score: 0.8155
S-score: 21098845.67
